# Processing of microglia two-photon time-lapse data

Pipeline used in the manuscript to go from raw Nikon `.nd2` stacks to per-timepoint microglia
masks and morphology tables:

1. Extract microglia candidates, masks and brain outlines.
2. Estimate rigid motion-correction transforms on the brain outline and apply them to the outline, microglia-mask and raw stacks.
3. Label and characterise motion-corrected microglia (per-timepoint morphology table).

See `README.md` for installation and the required environment variables.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import skimage as ski
import pandas as pd
import os
import dask.array as da

In [ ]:
# Configuration: set these environment variables before starting Jupyter (see README.md).
from microbiome.microglia import TEMP_DIR  # fast local scratch space ($MICROBIOME_TEMP_DIR)

GSHEET_KEYFILE = os.environ["GSHEET_KEYFILE"]  # path to a Google service-account JSON key
GSHEET_NAME = os.environ.get("GSHEET_NAME", "experiments")
GSHEET_WORKSHEET = os.environ.get("GSHEET_WORKSHEET", "diet_brain-microglia")
ANALYSIS_DIR = os.environ["ANALYSIS_DIR"]  # output directory for collection-level results

In [ ]:
import dask
dask.config.set({"distributed.worker.memory.terminate": 1.00})  # Restart if > 100% used
dask.config.set({"distributed.worker.memory.pause": 0.98})  # Pause if > 98% used
dask.config.set({'distributed.worker.memory.spill': 0.90})  # Spill to disk if > 70% used
dask.config.set({'distributed.worker.memory.target': 0.90})   # Target 50% usage

In [ ]:
from microbiome._vendor.analysis_toolbox.utils import now_str

In [ ]:
from microbiome._vendor.analysis_toolbox.fileio import load_gsheet

datasets_sheet = load_gsheet(
    keyfile_name=GSHEET_KEYFILE,
    sheet_name=GSHEET_NAME,
    worksheet=GSHEET_WORKSHEET
    )

datasets_sheet


In [ ]:
collection = ''

if collection == 'paper':

    drug_selection = np.logical_or.reduce([
        datasets_sheet['drug'] == 'unfed',
        # datasets_sheet['drug'] == 'Sparos', ## done
        # datasets_sheet['drug'] == 'para',  ## done
        # datasets_sheet['drug'] == 'yolk',  ## done
        # datasets_sheet['drug'] == 'PC',  ## done
        datasets_sheet['drug'] == 'PA', 
        datasets_sheet['drug'] == 'CLB', 
        # datasets_sheet['drug'] == 'PC+DMSO',  ## done
        # datasets_sheet['drug'] == 'PC+SB225002',   ## done
        # datasets_sheet['drug'] == 'yolk+DMSO',  ## done
        # datasets_sheet['drug'] == 'yolk+SB225002', ## done

        ])

else:

    drug_selection = np.ones(len(datasets_sheet), dtype=bool)




In [ ]:
selected_data = datasets_sheet[np.logical_and.reduce(
    [
        datasets_sheet['status'] =='processing',
        datasets_sheet['stacksize'] == '200um',
        drug_selection
        ]
        ) 
        ]

display(selected_data.shape)
selected_data

In [ ]:
data_dirs = selected_data['rootdir'].to_list()

data_labels = {
    'drug': selected_data['drug'].to_list(), 
    'slice': selected_data['slice'].to_list(), 
    }

In [ ]:
from microbiome._vendor.analysis_toolbox.dataorg.dataset_collection import initialize_dataset_collection

datasets = initialize_dataset_collection(data_dirs=data_dirs, collection=collection, setup='wee', analysis_dir=ANALYSIS_DIR,data_labels=data_labels)
datasets.open_datasets()
datasets.prepare_dir()
datasets.configure_funcim(im_raw=True, ephys=False, segmented=False, im_pro=False, dataset_type='nikon')

# 8m 29s for a stack with 101 planes and 48 time points
datasets.init_spim()

In [ ]:

for i, (spim_dset, dset) in tqdm(enumerate(zip(datasets.spim_dsets, datasets.dsets)), total=len(datasets.spim_dsets)):

    spim_dset.rig = 'nikon2p'
    print(f'Index: {i}')
    print(f'Dataset: {spim_dset.savepath}')
    print(f'Drug: {spim_dset.drug}')

    if hasattr(spim_dset, 'im_CM0_raw'):
        print(f'Raw Data Shape: {spim_dset.im_CM0_raw.data.shape}')
    else:
         print('does not have raw data.')

    print('')

In [ ]:
from microbiome._vendor.analysis_toolbox.parallel import dask_client_compute

background_subtract = False
debug = False

if background_subtract:

    for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

        spim_dset.im_CM0_raw_mean_stack_da = spim_dset.im_CM0_raw.data.mean(dim="T")
        spim_dset.im_CM0_raw_median_stack_da = spim_dset.im_CM0_raw.data.median(dim="T")

        if debug:
            spim_dset.im_CM0_raw_mean_stack = dask_client_compute(spim_dset.im_CM0_raw_mean_stack_da)
            spim_dset.im_CM0_raw_median_stack = dask_client_compute(spim_dset.im_CM0_raw_median_stack_da)

            from microbiome._vendor.analysis_toolbox.stacks_helper import plot_planes
            plot_planes(spim_dset.im_CM0_raw_mean_stack, percentile_min=0.1, percentile_max=99.9, figsize_scale=2)

        spim_dset.im_CM0_raw_bgsub_da = spim_dset.im_CM0_raw.data - spim_dset.im_CM0_raw_mean_stack_da 

In [ ]:
from microbiome.microglia import correct_timestamps

t_axis_mode = 'theoretical'  # 'empirical' or 'theoretical' or 'on average'

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

    print(f"Dataset {spim_dset.expt_date.strftime('%Y%m%d')} {spim_dset.fish_num}...")
    
    spim_dset.im_CM0_raw.events = pd.DataFrame(spim_dset.im_CM0_raw.metadata.events())
    spim_dset.im_CM0_raw.events['Timestamp [s]'] =  spim_dset.im_CM0_raw.events['Time [s]'] - spim_dset.im_CM0_raw.events.loc[0, 'Time [s]']
    spim_dset.im_CM0_raw.t_axis_s = np.rint(spim_dset.im_CM0_raw.events.groupby('T Index')['Timestamp [s]'].min()).astype('int').to_numpy()

    assert len(spim_dset.im_CM0_raw.t_axis_s) == spim_dset.im_CM0_raw.data.metadata['experiment'][0].count
    spim_dset.im_CM0_raw.t_axis_h = spim_dset.im_CM0_raw.t_axis_s / 3600

    print("Timestamps extracted")
    print(spim_dset.im_CM0_raw.t_axis_s)
    print(spim_dset.im_CM0_raw.t_axis_h)

    if hasattr(spim_dset.im_CM0_raw.metadata.experiment[0].parameters, 'periods'):

        print("Two epoch experiment detected!")

        manual_intervals = [0.063, 0.5]          # hours per segment
        manual_durations = [1.0, 96.0]         # total hours for each segment

        print(f"Periods: {spim_dset.im_CM0_raw.metadata.experiment[0].parameters.periods}")

        spim_dset.im_CM0_raw.t_axis_h = correct_timestamps(spim_dset.im_CM0_raw.t_axis_h, 
                                spim_dset.im_CM0_raw.metadata.experiment[0].parameters.periods,
                                manual_intervals_h=manual_intervals,
                                manual_durations_h=manual_durations)

        spim_dset.im_CM0_raw.t_axis_s = spim_dset.im_CM0_raw.t_axis_h * 3600

    else:
    
        print("Single epoch experiment detected!")

        if np.all(spim_dset.im_CM0_raw.t_axis_h == 0) or spim_dset.im_CM0_raw.t_axis_h[-1] > 72 or ((spim_dset.im_CM0_raw.t_axis_h[0] == 0) and (spim_dset.im_CM0_raw.t_axis_h[1] == 0)):

            print('Timestamps inaccurate!')

            if t_axis_mode == 'on average':
                spim_dset.im_CM0_raw.t_axis_s = np.arange(0,
                                                            (spim_dset.im_CM0_raw.metadata.experiment[0].parameters.durationMs / 1000),
                                                            (spim_dset.im_CM0_raw.metadata.experiment[0].parameters.periodDiff.avg / 1000))

                if len(spim_dset.im_CM0_raw.t_axis_s) != spim_dset.im_CM0_raw.data.metadata['experiment'][0].count:

                    spim_dset.im_CM0_raw.t_axis_s = np.linspace(0, 
                                                                (spim_dset.im_CM0_raw.metadata.experiment[0].parameters.durationMs / 1000),
                                                                spim_dset.im_CM0_raw.data.metadata['experiment'][0].count)


            elif t_axis_mode == 'theoretical':

                step_ms = spim_dset.im_CM0_raw.metadata.experiment[0].parameters.periodMs
                stop_ms = spim_dset.im_CM0_raw.metadata.experiment[0].parameters.durationMs

                print(f"Original step_ms: {step_ms}, stop_ms: {stop_ms}")

                if step_ms != 1800000.0:
                    print(f"Abberation with step_ms in {spim_dset.expt_id}! ({step_ms})")
                    step_ms = 1800000.0
                
                if stop_ms != 86400000.0 or 172800000.0 :
                    print(f"Abberation with stop_ms in {spim_dset.expt_id}! ({stop_ms})")
                    stop_ms = 172800000.0 

                spim_dset.im_CM0_raw.t_axis_s = np.arange(0,
                                                        (stop_ms/ 1000)+1,
                                                        (step_ms/ 1000))

                spim_dset.im_CM0_raw.t_axis_s = spim_dset.im_CM0_raw.t_axis_s[:spim_dset.im_CM0_raw.data.shape[0]]

        else:

            print("Timestamp seems valid!")


        assert len(spim_dset.im_CM0_raw.t_axis_s) == spim_dset.im_CM0_raw.data.metadata['experiment'][0].count
        spim_dset.im_CM0_raw.t_axis_h = spim_dset.im_CM0_raw.t_axis_s / 3600

    print("Final timestamps")
    print(spim_dset.im_CM0_raw.t_axis_s)
    print(spim_dset.im_CM0_raw.t_axis_h)

    print("")

In [ ]:

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):
    print(spim_dset.im_CM0_raw.t_axis_h)

In [ ]:
print(f'Voxel Spacing (z, y, x)')
for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):
    spim_dset.im_CM0_raw.vox_spacing = spim_dset.im_CM0_raw.data.metadata['metadata'].channels[0].volume.axesCalibration[::-1]
    print(spim_dset.im_CM0_raw.vox_spacing)

In [ ]:
from microbiome._vendor.analysis_toolbox.dataset.functional_imaging import load_func_im
import zarr

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

    ## (1) Look for whether a baseline dataset is defined
    if spim_dset.metadata_pd is not None:

        print(spim_dset.metadata_pd.loc['baseline', 'value'])

        if 'baseline' in spim_dset.metadata_pd.index:

            print(f"Loading baseline functional imaging data for {spim_dset.savepath}...")
            spim_dset.load_metadata()

            spim_dset.baseline_dset, spim_dset.baseline_spim_dset = load_func_im(
                spim_dset.metadata_pd.loc['baseline', 'value'], 
                '', 
                ephys=False, 
                im_raw=True, 
                segmented=False,
                im_pro=False, 
                channels='all', 
                dataset_type='nikon', 
                project='wee'
                )
            
            display(spim_dset.baseline_spim_dset.im_CM0_raw.data)

    ## (2) If a baseline dataset is found, extract microglia
    if hasattr(spim_dset, 'baseline_spim_dset'):
        spim_dset.baseline_spim_dset.vox_spacing = spim_dset.baseline_spim_dset.data.metadata['metadata'].channels[0].volume.axesCalibration[::-1]

        ## [TODO] OBSOLETE: use extract_or_load_microglia()
        spim_dset.baseline_spim_dset.small_obj_labels_path = f'{spim_dset.procpath}baseline_small_obj_labels.zarr'

        if not os.path.exists(spim_dset.baseline_spim_dset.small_obj_labels_path):

            spim_dset.baseline_spim_dset.small_obj_labels = extract_microglia(spim_dset.baseline_spim_dset.im_CM0_raw.data, spim_dset.baseline_spim_dset.vox_spacing)
            zarr.save_array(spim_dset.baseline_spim_dset.small_obj_labels_path, spim_dset.baseline_spim_dset.small_obj_labels)
            spim_dset.baseline_spim_dset.small_obj_labels = da.from_zarr(spim_dset.baseline_spim_dset.small_obj_labels_path, inline_array=True)

        else:
            spim_dset.baseline_spim_dset.small_obj_labels = da.from_zarr(spim_dset.baseline_spim_dset.small_obj_labels_path, inline_array=True)
            display(spim_dset.baseline_spim_dset.small_obj_labels)


In [ ]:
## TODO: automated way to compute num_workers
algo = 'recent'  ## 'legacy' or 'recent'
copy_data_to_local = True
recompute = False

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

    if algo == 'recent':

        from microbiome.microglia import extract_microglia_candidates_using_complex_outline_computation

        extract_microglia_candidates_using_complex_outline_computation(
            spim_dset,
            recompute=recompute,
            copy_data_to_local=copy_data_to_local,
        )

    elif algo == 'legacy':

        from microbiome.microglia import extract_microglia_candidates_using_simple_outline_computation

        extract_microglia_candidates_using_simple_outline_computation(
            spim_dset,
            recompute=recompute
        )

In [ ]:
## 22 mins 
from bigstream import motion_correct
import SimpleITK as sitk
import zarr
import shutil

recompute = False

rigid_kwargs = {
    'metric': 'MS',
    'alignment_spacing': 1.0,
    'sampling': 'RANDOM',
    'sampling_percentage': 0.2,
    'optimizer': 'GD',
    'shrink_factors':[1], 
    'smooth_sigmas':[0],
    'initial_condition': 'CENTER',
    'optimizer_args':{
        'learningRate': 1, ## 0.25
        'numberOfIterations':1000,  ## 100
        'estimateLearningRate': sitk.ImageRegistrationMethod.Once
    },
}

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets), position=0):

    if recompute:
        
        if os.path.exists(spim_dset.procpath + f'motioncorrect_transforms.zarr'):
            print("motioncorrect_transforms.zarr already exists. Removing...")
            shutil.rmtree(spim_dset.procpath + f'motioncorrect_transforms.zarr')

    if os.path.exists(spim_dset.procpath + f'motioncorrect_transforms.zarr'):
        print('Transforms already exist, loading...')
        spim_dset.transforms = zarr.open(spim_dset.procpath + f'motioncorrect_transforms.zarr', mode='r')
    else:

        from microbiome._vendor.analysis_toolbox.fileio import make_local_copy
        local_path = make_local_copy(spim_dset.brain_outline_zarr_path, TEMP_DIR)

        spim_dset.moving_stacks = zarr.open(local_path, mode='r')
        spim_dset.fixed_stack = spim_dset.moving_stacks[0]  ## ~3 mins

        if spim_dset.im_CM0_raw.vox_spacing[0] == 1:
            num_workers = 8  ## 8 works for most
        elif spim_dset.im_CM0_raw.vox_spacing[0] == 5:
            num_workers = 24

        spim_dset.transforms = motion_correct.motion_correct(
            spim_dset.fixed_stack,
            spim_dset.moving_stacks,
            target_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            timeseries_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            indices=None,
            mask=np.full(spim_dset.fixed_stack.shape, True, dtype=bool),
            steps=[('rigid', rigid_kwargs)],
            cluster_kwargs={'cluster_type': 'local_cluster', 'n_workers': num_workers, 'threads_per_worker': 1, 'memory_limit': '6GB'},
            write_path=spim_dset.procpath + f'motioncorrect_transforms.zarr',
        )

In [ ]:
# from microbiome._vendor.analysis_toolbox.utils import sizeof_fmt
# sizeof_fmt(spim_dset.im_CM0_raw.data_local_zarr.nbytes / spim_dset.im_CM0_raw.data_local_zarr.shape[0])

In [ ]:
%%time
recompute = False

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets), position=0):
    
    if recompute:
        
        if os.path.exists(spim_dset.procpath + f'motion_corrected.zarr'):
            shutil.rmtree(spim_dset.procpath + f'motion_corrected.zarr')

    if os.path.exists(spim_dset.procpath + f'motion_corrected.zarr'):
        print('Motion corrected frames already exist, loading...')
        spim_dset.aligned_frames = zarr.open(spim_dset.procpath + f'motion_corrected.zarr', mode='r')
    else:

        spim_dset.moving_stacks = zarr.open(spim_dset.brain_outline_zarr_path, mode='r')
        spim_dset.fixed_stack = spim_dset.moving_stacks[0]  ## ~3 mins
        
        if spim_dset.im_CM0_raw.vox_spacing[0] == 1:
            num_workers = 12
        elif spim_dset.im_CM0_raw.vox_spacing[0] == 5:
            num_workers = 24

        spim_dset.aligned_frames = motion_correct.resample_frames(
            spim_dset.fixed_stack,
            spim_dset.moving_stacks,
            target_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            timeseries_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            transform_list=[spim_dset.transforms],
            mask=np.full(spim_dset.fixed_stack.shape, True, dtype=bool),
            interpolator='0',
            cluster_kwargs={'cluster_type': 'local_cluster', 'n_workers': num_workers, 'threads_per_worker': 1, 'memory_limit': '2GB'},
            write_path=spim_dset.procpath + f'motion_corrected.zarr',
        )

In [ ]:
%%time
from bigstream import motion_correct

recompute = False

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets), position=0):

    spim_dset.microglia_mask_zarr = zarr.open(spim_dset.microglia_mask_zarr_path, mode='r')
    spim_dset.motion_corrected_microglia_mask_zarr_path = spim_dset.procpath + f'motion_corrected_microglia_mask.zarr'

    if not hasattr(spim_dset, 'transforms'):
        spim_dset.transforms = zarr.open(spim_dset.procpath + f'motioncorrect_transforms.zarr', mode='r')
    
    if recompute:
        
        if os.path.exists(spim_dset.procpath + f'motion_corrected_microglia_mask.zarr'):
            shutil.rmtree(spim_dset.procpath + f'motion_corrected_microglia_mask.zarr')

    if os.path.exists(spim_dset.motion_corrected_microglia_mask_zarr_path):
        print('Motion corrected frames already exist, loading...')
        spim_dset.aligned_frames = zarr.open(spim_dset.motion_corrected_microglia_mask_zarr_path, mode='r')
    else:

        spim_dset.moving_stacks = zarr.open(spim_dset.brain_outline_zarr_path, mode='r')
        spim_dset.fixed_stack = spim_dset.moving_stacks[0]  ## ~3 mins

        if spim_dset.im_CM0_raw.vox_spacing[0] == 1:
            num_workers = 12
        elif spim_dset.im_CM0_raw.vox_spacing[0] == 5:
            num_workers = 24


        spim_dset.aligned_frames = motion_correct.resample_frames(
            spim_dset.fixed_stack,
            spim_dset.microglia_mask_zarr,
            target_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            timeseries_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            transform_list=[spim_dset.transforms],
            mask=np.full(spim_dset.fixed_stack.shape, True, dtype=bool),
            interpolator='0',
            cluster_kwargs={'cluster_type': 'local_cluster', 'n_workers': num_workers, 'threads_per_worker': 1, 'memory_limit': '2GB'},
            write_path=spim_dset.motion_corrected_microglia_mask_zarr_path,
        )

In [ ]:
%%time
from bigstream import motion_correct
recompute = False

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets), position=0):

    spim_dset.im_CM0_raw_rechunked_path = f'{spim_dset.procpath}im_CM0_raw_rechunked.zarr'
    spim_dset.im_CM0_raw_rechunked = zarr.open(spim_dset.im_CM0_raw_rechunked_path, mode='r')
    spim_dset.motion_corrected_im_CM0_raw_path = spim_dset.procpath + f'motion_corrected_im_CM0_raw.zarr'

    if not hasattr(spim_dset, 'transforms'):
        spim_dset.transforms = zarr.open(spim_dset.procpath + f'motioncorrect_transforms.zarr', mode='r')
    
    if recompute:
        
        if os.path.exists(spim_dset.motion_corrected_im_CM0_raw_path):
            shutil.rmtree(spim_dset.motion_corrected_im_CM0_raw_path)

    if os.path.exists(spim_dset.motion_corrected_im_CM0_raw_path):
        print('Motion corrected frames already exist, loading...')
        spim_dset.aligned_frames = zarr.open(spim_dset.motion_corrected_im_CM0_raw_path, mode='r')
    else:

        spim_dset.moving_stacks = zarr.open(spim_dset.brain_outline_zarr_path, mode='r')
        spim_dset.fixed_stack = spim_dset.moving_stacks[0]  ## ~3 mins

        if spim_dset.im_CM0_raw.vox_spacing[0] == 1:
            num_workers = 12
        elif spim_dset.im_CM0_raw.vox_spacing[0] == 5:
            num_workers = 24


        spim_dset.aligned_frames = motion_correct.resample_frames(
            spim_dset.fixed_stack,
            spim_dset.im_CM0_raw_rechunked,
            target_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            timeseries_spacing=np.array(spim_dset.im_CM0_raw.vox_spacing),
            transform_list=[spim_dset.transforms],
            mask=np.full(spim_dset.fixed_stack.shape, True, dtype=bool),
            interpolator='1',
            cluster_kwargs={'cluster_type': 'local_cluster', 'n_workers': num_workers, 'threads_per_worker': 1, 'memory_limit': '2GB'},
            write_path=spim_dset.motion_corrected_im_CM0_raw_path,
        )

In [ ]:
from microbiome.microglia import extract_and_characterize_microglia
from microbiome._vendor.analysis_toolbox.fileio import chunk_and_saveas_zarr
import shutil

recompute = False

for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

    spim_dset.motion_corrected_microglia_mask_zarr_path = spim_dset.procpath + f'motion_corrected_microglia_mask.zarr'
    spim_dset.motion_corrected_microglia_mask_zarr = zarr.open(spim_dset.motion_corrected_microglia_mask_zarr_path, mode='r')

    spim_dset.motion_corrected_microglia_mask_df_path = spim_dset.procpath + f'microglia_df.pkl'
    spim_dset.motion_corrected_microglia_mask_labels_path = spim_dset.procpath + f'microglia_labels.zarr'

    if recompute: 

        if os.path.exists(spim_dset.motion_corrected_microglia_mask_df_path):
            os.remove(spim_dset.motion_corrected_microglia_mask_df_path)

        if os.path.exists(spim_dset.motion_corrected_microglia_mask_labels_path):
            shutil.rmtree(spim_dset.motion_corrected_microglia_mask_labels_path)

    spim_dset.microglia_mask_df =\
         extract_and_characterize_microglia(
            spim_dset, 
            spim_dset.motion_corrected_microglia_mask_zarr,
            labels_savepath=spim_dset.motion_corrected_microglia_mask_labels_path,
            microglia_df_savepath=spim_dset.motion_corrected_microglia_mask_df_path
            )

In [ ]:
for spim_dset, dset in tqdm(zip(datasets.spim_dsets, datasets.dsets), total=len(datasets.spim_dsets)):

    print(f"Dataset {spim_dset.expt_date.strftime('%Y%m%d')} {spim_dset.fish_num}...")

    print(spim_dset.microglia_mask_df['time_s'].unique())
    spim_dset.microglia_mask_df['time_h'] = spim_dset.microglia_mask_df['time_s'] / 3600

    print(spim_dset.microglia_mask_df['time_h'].unique())